In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")

In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import features

# Window size: 10k samples = 0.4ms duration
CHUNK_SIZE = 10000

# Subsampling: Take 5,000 chunks per file (50M samples used per file)
CHUNKS_PER_FILE = 5000

print(f"Environment Linked.")
print(f"Input Directory: {JAMSHIELD_NPY_DIR}")
print(f"Output File:     {MACHINE_A_DATASET_FILE}")
print(f"Active Features: {MACHINE_A_FEATURES}")


In [ ]:
try:
    meta_df = pd.read_csv(JAMSHIELD_METADATA_FILE)
    
    # Create Fast Lookup Dictionary
    # Key: (filename, scenario) -> {Data}
    meta_lookup = {}
    
    for _, row in meta_df.iterrows():
        fname = str(row['filename']).strip()   # 'w1'
        scenario = str(row['scenario']).strip() # 'Gauss'
        
        key = (fname, scenario)
        meta_lookup[key] = {
            'jamming_power': row['jamming_power'],
            'distance': row['distance']
        }
        
    print(f"Metadata loaded successfully ({len(meta_df)} rows).")

except Exception as e:
    print(f"WARNING: Could not load metadata. ({e})")
    meta_lookup = {}

def get_file_metadata(filename):
    """
    Parses filename (e.g., 'w1_barrage.npy') and retrieves physics info.
    """
    # Parse Filename
    parts = filename.stem.split('_') 
    file_id = parts[0]   # w1
    scenario_part = parts[1]  # barrage, clean, sine
    
    # Map Filename parts to CSV 'scenario' column
    type_map = {
        'clean': 'No',
        'nojamming': 'No',
        'barrage': 'Gauss',
        'gaussian': 'Gauss',
        'sine': 'Sin'
    }
    
    excel_type = type_map.get(scenario_part.lower())
    
    # Retrieve
    if excel_type:
        data = meta_lookup.get((file_id, excel_type))
        if data:
            return {
                'jamming_power': data['jamming_power'],
                'distance': data['distance'],
                'jammer_type': excel_type
            }

    return {'jamming_power': -1, 'distance': -1, 'jammer_type': 'Unknown'}

In [ ]:
def process_real_files():
    dataset = []
    
    # Find all NPY files
    patterns = ["*_clean.npy", "*_sine.npy", "*_gaussian.npy"]
    files = []
    for p in patterns:
        files.extend(JAMSHIELD_NPY_DIR.glob(p))
    files = sorted(list(set(files))) 
    
    print(f"Found {len(files)} Real Hardware files to process.")
    
    for f_path in tqdm(files, desc="Extracting Features"):
        try:
            fname = f_path.name
            
            # Determine Label (0=Clean, 1=Jamming)
            # Sine AND Barrage/Gaussian count as Jamming
            is_clean = "clean" in fname.lower() or "nojamming" in fname.lower()
            label = 0 if is_clean else 1
            
            # Get Metadata
            meta = get_file_metadata(f_path)
            
            # Load Data (Memory Mapped)
            data = np.load(f_path, mmap_mode='r')
            
            # Subsample
            total_samples = len(data)
            max_chunks = total_samples // CHUNK_SIZE
            
            if max_chunks > CHUNKS_PER_FILE:
                indices = np.random.choice(max_chunks, CHUNKS_PER_FILE, replace=False)
                indices.sort()
            else:
                indices = range(max_chunks)
            
            # Extract Loop
            for i in indices:
                # Load chunk into RAM
                chunk = np.array(data[i*CHUNK_SIZE : (i+1)*CHUNK_SIZE])
                
                # --- MODULAR CALL ---
                # Calculate only what is in config.MACHINE_A_FEATURES
                row = features.compute_feature_vector(chunk, MACHINE_A_FEATURES)
                # --------------------
                
                # Enrich with Labels & Metadata
                row['label'] = label
                row['source'] = 'real_hardware'
                row['filename'] = fname
                row['jamming_power'] = meta['jamming_power']
                row['distance'] = meta['distance']
                row['jammer_type'] = meta['jammer_type']
                
                dataset.append(row)
                
        except Exception as e:
            print(f"Error processing {fname}: {e}")
            
    return dataset

In [ ]:
full_data = process_real_files()

df = pd.DataFrame(full_data)

# Clean & Shuffle
if df.isnull().values.any():
    print(f"Dropped {df.isnull().sum().max()} rows containing NaNs.")
    df = df.dropna()

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save
df.to_csv(MACHINE_A_DATASET_FILE, index=False)

print("="*60)
print(f"Dataset Saved Successfully!")
print(f"Location: {MACHINE_A_DATASET_FILE}")
print(f"Total Rows: {len(df)}")
print("-" * 30)
print("Class Distribution:")
print(df['label'].value_counts())
print("-" * 30)
print(df.head())